### Databricks client (python)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam, catalog

w = WorkspaceClient()

# 1. Create a New User
new_user = w.users.create(
    user_name="analyst_user@example.com",
    display_name="Analyst User"
)
print(f"Created user: {new_user.display_name} (ID: {new_user.id})")

In [0]:
# 2. Create a New Group
analyst_group = w.groups.create(
    display_name="Data_Analysts_Team"
)
print(f"Created group: {analyst_group.display_name} (ID: {analyst_group.id})")

In [0]:
# 3. Add User to the Group
w.groups.patch(
    id=analyst_group.id,
    operations=[
        iam.Patch(
            op=iam.PatchOp.ADD,
            path="members",
            value=[{"value": new_user.id}]
        )
    ]
)
print(f"Added {new_user.user_name} to {analyst_group.display_name}")

In [0]:
# 4. Assign Permissions in Unity Catalog
# Granting USE_CATALOG and USE_SCHEMA is required for data access
catalog_name = "end_to_end_retail"
schema_name = "db_gold"
table_name = "tbg_customer_address_clustering"

w.grants.update(
    full_name=catalog_name,
    securable_type=catalog.SecurableType.CATALOG,
    changes=[
        catalog.PermissionsChange(
            principal=analyst_group.display_name,
            add=[catalog.Privilege.USE_CATALOG]
        )
    ]
)

In [0]:
# Grant SELECT access to a specific table
w.grants.update(
    full_name=f"{catalog_name}.{schema_name}.{table_name}",
    securable_type=catalog.SecurableType.TABLE,
    changes=[
        catalog.PermissionsChange(
            principal=analyst_group.display_name,
            add=[catalog.Privilege.SELECT]
        )
    ]
)
print(f"Granted SELECT on {table_name} to {analyst_group.display_name}")

#### Grant (SQL)

In [0]:
# Managing permissions for a specific group
def grant_read_access(catalog_name, principal):
    spark.sql(f"GRANT USAGE ON CATALOG {catalog_name} TO `{principal}`")
    spark.sql(f"GRANT USAGE ON SCHEMA {catalog_name}.default TO `{principal}`")
    spark.sql(f"GRANT SELECT ON SCHEMA {catalog_name}.default TO `{principal}`")
    print(f"Read access granted to {principal}")

grant_read_access("gold_layer", "data_analysts_group")


#### Encryption function

- end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched

In [0]:
%sql
SELECT email FROM end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched limit 1

In [0]:
%sql
CREATE OR REPLACE FUNCTION email_mask(email STRING)
  RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin') THEN email
    ELSE sha2(email, 256)
  END;

In [0]:
%sql
-- 2. Apply it to a table (SQL)
ALTER TABLE end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched 
ALTER COLUMN email SET MASK email_mask;

#### Table extended functions

In [0]:
# Discovering metadata
table_info = spark.sql("DESCRIBE TABLE EXTENDED end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched")

# Displaying table comments to ensure data discovery documentation exists
display(table_info.filter(table_info.col_name == "Comment"))

# Checking Table Constraints
constraints = spark.sql("SHOW TBLPROPERTIES end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched")
display(constraints)


### 1. Python Script: Tagging PII, PHI, owner

In [0]:
# This helps data stewards find PII quickly
def tag_tables_with_pii(table_list):
    for table in table_list:
        spark.sql(f"ALTER TABLE {table} SET TAGS ('contains_pii' = 'true', 'data_sensitivity' = 'high')")

# Example usage:
# tag_tables_with_pii(["main.sensitive_data.raw_users", "main.sensitive_data.customer_info"])

#### Script: Applying Multi-Level Tags
- example

In [0]:
# 1. Tagging a Schema (Broad Governance)
spark.sql("""
  ALTER SCHEMA main.healthcare_data 
  SET TAGS ('data_classification' = 'PHI', 'owner' = 'clinical_ops')
""")

# 2. Tagging a Table (Operational Metadata)
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  SET TAGS (
    'refresh_frequency' = 'Daily',
    'contains_pii' = 'true',
    'retention_period' = '7_years'
  )
""")

# 3. Tagging a Column (Fine-Grained Discovery)
spark.sql("""
  ALTER TABLE main.healthcare_data.patient_logs 
  ALTER COLUMN patient_ssn SET TAGS ('pii_type' = 'ssn', 'masking_applied' = 'true')
""")

### 2. Defining Secure Encryption

In [0]:
%sql
-- Create a schema for governance logic
CREATE SCHEMA IF NOT EXISTS end_to_end_retail.governance_functions;

#### sha2 encryption

In [0]:
%sql
-- Email 
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.email_mask(email STRING)
  RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('hr_admins') THEN email
    ELSE sha2(email, 256)
  END;

### 3. Row-Level Security (Data Filtering)
- example

In [0]:
%sql
-- Create a function to filter rows by region
CREATE OR REPLACE FUNCTION governance_functions.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', region)) OR is_account_group_member('admin');

-- Apply the filter to the table
ALTER TABLE main.sensitive_data.patient_records 
SET ROW FILTER governance_functions.region_filter ON (region);


## Phase 1: Encryption Functions
- example

In [0]:
%sql
-- Email
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.email_mask(email STRING)
  RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('hr_admins') THEN email
    ELSE sha2(email, 256)
  END;

-- Credit Card
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.cc_mask(cc_mask STRING)
  RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('finance_admins') THEN cc_mask
    ELSE sha2(cc_mask, 256)
  END;

-- PII
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.pii_redact(ssn STRING)
RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('medical_staff') THEN ssn
    ELSE sha2(ssn, 256)
  END;

--- ROW FILTER: Regional Governance
CREATE OR REPLACE FUNCTION governance_root.policies.region_filter(region STRING)
RETURN is_account_group_member(concat('group_', lower(region))) OR is_account_group_member('admin');

## Phase 2: Create Principals & Assign Permissions
- example

In [0]:
%sql
-- Public/General Access
GRANT USAGE ON CATALOG end_to_end_retail TO `all_users`;

-- Sensitive Data Access (Restricted to specific principals)
GRANT USAGE ON SCHEMA end_to_end_retail.example_table TO `hr_admins`;
GRANT SELECT ON SCHEMA end_to_end_retail.example_table TO `hr_admins`;

-- Finance Access
GRANT USAGE ON SCHEMA end_to_end_retail.example_table TO `finance_admin`;
GRANT SELECT ON SCHEMA end_to_end_retail.example_table TO `finance_admin`;


## Phase 3: Robust Tagging & Policy Application Script
- Example tagging, row filtering, and column encryption;

In [0]:
from pyspark.sql import SparkSession

def apply_governance(table_fqn, classification, region_col=None):
    """
    Applies a 3-layer security model to a table.
    1. Layer: Tagging (Discovery)
    2. Layer: Row Filtering (Redaction by Region)
    3. Layer: Column encryption (PII Protection)
    """
    
    print(f"--- Applying Governance to {table_fqn} [{classification}] ---")

    # LAYER 1: TAGGING
    spark.sql(f"ALTER TABLE {table_fqn} SET TAGS ('classification' = '{classification}', 'governance_synced' = 'true')")

    # LAYER 2 & 3: SENSITIVITY-BASED POLICIES
    if classification in ["PHI", "PII_HIGH"]:
        # Apply Row Level Redaction if a region column is provided
        if region_col:
            spark.sql(f"ALTER TABLE {table_fqn} SET ROW FILTER governance_root.policies.region_filter ON ({region_col})")
            print(f"Row filter applied on {region_col}")

        # Identify columns to mask/redact based on standard naming conventions
        cols = spark.table(table_fqn).columns
        
        for col in cols:
            if "email" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK end_to_end_retail.governance_functions.email_mask")
            
            elif "credit_card" in col.lower() or "cc_num" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK end_to_end_retail.governance_functions.cc_mask")
                
            elif "ssn" in col.lower() or "health_id" in col.lower():
                spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN {col} SET MASK end_to_end_retail.governance_functions.phi_redact")
                
    print(f"Governance sync complete for {table_fqn}.\n")

# --- EXECUTION EXAMPLE ---
apply_governance("main.healthcare.patient_records", "PHI", region_col="data_region")

apply_governance("main.finance.transactions", "PII_HIGH")


### Policy
- Catalog, Schema, or Table level;

In [0]:
%sql
-- Create the filter function first
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.filter_by_city(city STRING)
RETURN city = 'London' OR is_account_group_member('admin');

In [0]:
%sql
CREATE OR REPLACE POLICY `userExampleCityFilter`
ON TABLE end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched
COMMENT 'user example city filter'
ROW FILTER end_to_end_retail.governance_functions.filter_by_city
TO `userExample`
FOR TABLES
MATCH COLUMNS hasTag('pii', 'Tag.Example') AS tagExample
USING COLUMNS (tagExample)